In [7]:
import pandas as pd
import numpy as np
import librosa
from tqdm import tqdm
import warnings
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
warnings.filterwarnings('ignore')

In [2]:
clip_df = pd.read_csv("../clips_metadata.csv",sep=",")

In [3]:
clip_df

,clip_filename,species,source_file,path
0,cangoo_0.wav,cangoo,SSW_001_20170225_010000Z.flac,../sliced_clips\cangoo_0.wav
1,cangoo_1.wav,cangoo,SSW_001_20170225_010000Z.flac,../sliced_clips\cangoo_1.wav
2,cangoo_2.wav,cangoo,SSW_001_20170225_010000Z.flac,../sliced_clips\cangoo_2.wav
3,cangoo_3.wav,cangoo,SSW_001_20170225_010000Z.flac,../sliced_clips\cangoo_3.wav
4,cangoo_4.wav,cangoo,SSW_001_20170225_010000Z.flac,../sliced_clips\cangoo_4.wav
...,...,...,...,...
49677,amecro_49708.wav,amecro,SSW_285_20170826_220029Z.flac,../sliced_clips\amecro_49708.wav
49678,amecro_49709.wav,amecro,SSW_285_20170826_220029Z.flac,../sliced_clips\amecro_49709.wav
49679,amecro_49710.wav,amecro,SSW_285_20170826_220029Z.flac,../sliced_clips\amecro_49710.wav
49680,amecro_49711.wav,amecro,SSW_285_20170826_220029Z.flac,../sliced_clips\amecro_49711.wav


In [4]:
def extract_features(file_path):
    y, sr = librosa.load(file_path, sr=None)

    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfcc_mean = np.mean(mfccs, axis=1)
    mfcc_std = np.std(mfccs, axis=1)

    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    chroma_mean = np.mean(chroma, axis=1)

    spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    centroid_mean = np.mean(spectral_centroid)

    zcr = librosa.feature.zero_crossing_rate(y)
    zcr_mean = np.mean(zcr)

    return np.hstack([mfcc_mean, mfcc_std, chroma_mean, centroid_mean, zcr_mean])

In [5]:
features = []
labels = []
failed_features = []

for idx, row in tqdm(clip_df.iterrows(), total=len(clip_df)):
    file_path = row['path']
    try:
        feat = extract_features(file_path)
        features.append(feat)
        labels.append(row['species'])
    except Exception as e:
        failed_features.append({'idx': idx, 'path': file_path, 'error': str(e)})
        continue

X = np.array(features)
y = np.array(labels)

np.save('../X_mfcc.npy', X)
np.save('../y_labels.npy',y)
print('Features saved — safe to reload anytime')

failed_df = pd.DataFrame(failed_features)
failed_df.to_csv('../failed_features.csv', index=False)
print(f'Failed: {len(failed_features)}')

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

100%|██████████| 49682/49682 [1:19:04<00:00, 10.47it/s]  


Features saved — safe to reload anytime
Failed: 4
X shape: (49678, 40)
y shape: (49678,)


In [ ]:

le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# SMOTE needs at least k_neighbors+1 samples in the smallest class
# default k_neighbors=5, your smallest class (moudov) has ~84 in train split — fine

pipeline = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ('svm', SVC(kernel='rbf', C=10, gamma='scale', probability=True, random_state=42))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)